# 07 Dashboard Export Assets

Purpose: prepare compact tables that the Streamlit dashboard can load quickly and reviewers can inspect.

In [1]:
from __future__ import annotations

import json
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import PROCESSED_DIR, TABLE_DIR, FIGURE_DIR, CPSC_DIR, PMPM_PROXY_REVENUE

for path in [PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")

Project root: D:\Project 1\rcm-cms-mvp


In [2]:
forecast = pd.read_csv(PROCESSED_DIR / "forecast_results.csv", parse_dates=["ds"])
metrics = pd.read_csv(TABLE_DIR / "forecast_metrics.csv")
state_growth = pd.read_csv(TABLE_DIR / "ma_scp_state_growth.csv")
pa = pd.read_csv(PROCESSED_DIR / "pa_predictions.csv")
delay = pd.read_csv(PROCESSED_DIR / "delay_predictions.csv")

best_models = metrics.sort_values(["target", "mape"]).groupby("target").head(1)
dashboard_forecast_summary = forecast[forecast["period_type"].isin(["holdout", "future"])].copy()
dashboard_state_opportunities = state_growth.head(30).copy()
dashboard_pa_summary = (
    pa.groupby(["procedure_type", "payer_type", "hybrid_risk_bucket"], as_index=False)
    .agg(cases=("hybrid_denial_risk", "size"), avg_denial_risk=("hybrid_denial_risk", "mean"))
    .sort_values("avg_denial_risk", ascending=False)
)
dashboard_delay_summary = (
    delay.groupby(["procedure_type", "payer_type", "risk_bucket"], as_index=False)
    .agg(combinations=("delay_risk_score", "size"), avg_delay_risk=("delay_risk_score", "mean"), delay_flags=("delay_flag", "sum"))
    .sort_values(["delay_flags", "avg_delay_risk"], ascending=False)
)

dashboard_forecast_summary.to_csv(TABLE_DIR / "dashboard_forecast_summary.csv", index=False)
dashboard_state_opportunities.to_csv(TABLE_DIR / "dashboard_state_opportunities.csv", index=False)
dashboard_pa_summary.to_csv(TABLE_DIR / "dashboard_pa_summary.csv", index=False)
dashboard_delay_summary.to_csv(TABLE_DIR / "dashboard_delay_summary.csv", index=False)
best_models.to_csv(TABLE_DIR / "dashboard_best_models.csv", index=False)

display(best_models)
display(dashboard_state_opportunities.head())
display(dashboard_pa_summary.head())
display(dashboard_delay_summary.head())

,model,target,mae,rmse,mape,holdout_months
0,linear_drift,observed_enrollment,1.735513e+04,2.217508e+04,0.000482,3
5,linear_drift,proxy_revenue,1.995840e+06,2.550134e+06,0.000482,3


,state,first_month,last_month,first_enrollment,last_enrollment,avg_enrollment,min_enrollment,max_enrollment,avg_mom_growth_pct,volatility_mom_growth_pct,counties,absolute_growth,growth_pct,opportunity_rank
0,CA,2024-01,2026-05,3440093.0,3718525.0,3.600701e+06,3440093.0,3718525.0,0.002790,0.003548,58,278432.0,0.080937,1
1,TX,2024-01,2026-05,2510805.0,2748087.0,2.625048e+06,2510805.0,2748087.0,0.003233,0.002550,254,237282.0,0.094504,2
2,FL,2024-01,2026-05,2853027.0,3058222.0,2.958052e+06,2853027.0,3058222.0,0.002486,0.002403,67,205195.0,0.071922,3
3,NY,2024-01,2026-05,1985636.0,2169002.0,2.082661e+06,1976532.0,2169002.0,0.003264,0.014590,62,183366.0,0.092346,4
4,NC,2024-01,2026-05,1205103.0,1344776.0,1.276969e+06,1205103.0,1344776.0,0.003930,0.003469,100,139673.0,0.115901,5


,procedure_type,payer_type,hybrid_risk_bucket,cases,avg_denial_risk
19,part_b_drug,MA_HMO,high,1,0.738568
4,DME,MA_HMO,high,2,0.715535
12,SNF,MA_PPO,high,2,0.712687
22,post_acute,MAPD,high,1,0.701484
14,home_health,MAPD,high,1,0.696001


,procedure_type,payer_type,risk_bucket,combinations,avg_delay_risk,delay_flags
14,SNF,MA_HMO,high,2,1.601862,2
13,SNF,MAPD,high,2,1.447422,2
28,post_acute,MAPD,high,1,1.748372,1
15,SNF,MA_PPO,high,1,1.680408,1
32,post_acute,MA_PPO,high,1,1.510672,1


## Dashboard Asset Decision

The dashboard should load compact CSV summaries plus the forecast result table. The full long MA SCP table stays as Parquet for modeling, not dashboard rendering.